In [2]:
# ================================
# IMPORTS
# ================================
import mlflow
import mlflow.pytorch
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ================================
# CONFIG
# ================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

# ================================
# DADOS
# ================================
dados = pd.read_csv('../data/processed/Customer-Churn-processed.csv')

X = dados.drop('Churn', axis=1).values
y = dados['Churn'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# convertendo para tensor
X_train = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1).to(DEVICE)

X_test = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1,1).to(DEVICE)

# ================================
# MODELO MLP
# ================================
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# ================================
# TREINO COM EARLY STOPPING
# ================================
def train_model(model, patience=10, epochs=100):

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    best_loss = np.inf
    patience_counter = 0

    for epoch in range(epochs):

        model.train()
        optimizer.zero_grad()

        outputs = model(X_train)
        loss = criterion(outputs, y_train)

        loss.backward()
        optimizer.step()

        # early stopping
        if loss.item() < best_loss:
            best_loss = loss.item()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping na época {epoch}")
            break

    return model

# ================================
# MÉTRICAS
# ================================
def evaluate_model(model):

    model.eval()
    with torch.no_grad():
        probs = model(X_test).cpu().numpy()
    
    y_pred = (probs > 0.5).astype(int)

    y_true = y_test.cpu().numpy()

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc = roc_auc_score(y_true, probs)
    pr = average_precision_score(y_true, probs)

    # MATRIZ CONFUSÃO
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    # MÉTRICA DE NEGÓCIO (trade-off)
    custo_fp = 100   # oferecer retenção desnecessária
    custo_fn = 500   # perder cliente

    custo_total = (fp * custo_fp) + (fn * custo_fn)

    return acc, f1, roc, pr, custo_total, fp, fn

# ================================
# MLFLOW
# ================================
mlflow.set_experiment("churn_mlp")

with mlflow.start_run(run_name="MLP_PyTorch"):

    model = MLP(X_train.shape[1]).to(DEVICE)

    model = train_model(model)

    acc, f1, roc, pr, custo, fp, fn = evaluate_model(model)

    # log
    mlflow.log_param("model", "MLP")
    mlflow.log_param("hidden_layers", [64,32])
    mlflow.log_param("activation", "ReLU")

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc)
    mlflow.log_metric("pr_auc", pr)
    mlflow.log_metric("business_cost", custo)

    mlflow.log_metric("false_positive", fp)
    mlflow.log_metric("false_negative", fn)

    mlflow.pytorch.log_model(model, "mlp_model")

    print("\n=== RESULTADOS MLP ===")
    print("Accuracy:", acc)
    print("F1:", f1)
    print("ROC-AUC:", roc)
    print("PR-AUC:", pr)
    print("Custo Total:", custo)

2026/04/06 19:13:22 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/06 19:13:22 INFO mlflow.store.db.utils: Updating database tables
2026/04/06 19:13:24 INFO mlflow.tracking.fluent: Experiment with name 'churn_mlp' does not exist. Creating a new experiment.
2026/04/06 19:13:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/06 19:13:26 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.



=== RESULTADOS MLP ===
Accuracy: 0.7885024840312278
F1: 0.569364161849711
ROC-AUC: 0.8323748998940814
PR-AUC: 0.6083371364069813
Custo Total: 100600


In [3]:
# ================================
# IMPORTS
# ================================
import os

import mlflow
import mlflow.pytorch
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ================================
# CONFIG GLOBAL MLFLOW (CORREÇÃO)
# ================================
BASE_DIR = os.path.abspath("..")  # sobe para raiz do projeto

mlflow.set_tracking_uri(f"sqlite:///{BASE_DIR}/mlflow.db")
mlflow.set_registry_uri(f"sqlite:///{BASE_DIR}/mlflow.db")

mlflow.set_experiment("churn_project")  # mesmo projeto

# ================================
# CONFIG TORCH
# ================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

# ================================
# CARREGAR DADOS (PROCESSADOS)
# ================================
dados = pd.read_csv(f"{BASE_DIR}/data/processed/Customer-Churn-processed.csv")

X = dados.drop('Churn', axis=1).values
y = dados['Churn'].values

# ================================
# PREPROCESSAMENTO
# ================================
scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ================================
# TENSORES
# ================================
X_train = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1).to(DEVICE)

X_test = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1,1).to(DEVICE)

# ================================
# MODELO MLP
# ================================
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# ================================
# TREINO COM EARLY STOPPING
# ================================
def train_model(model, epochs=100, patience=10):

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    best_loss = np.inf
    patience_counter = 0

    for epoch in range(epochs):

        model.train()
        optimizer.zero_grad()

        outputs = model(X_train)
        loss = criterion(outputs, y_train)

        loss.backward()
        optimizer.step()

        if loss.item() < best_loss:
            best_loss = loss.item()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping na época {epoch}")
            break

    return model

# ================================
# MÉTRICAS + TRADE-OFF
# ================================
def evaluate_model(model):

    model.eval()
    with torch.no_grad():
        probs = model(X_test).cpu().numpy()

    y_pred = (probs > 0.5).astype(int)
    y_true = y_test.cpu().numpy()

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc = roc_auc_score(y_true, probs)
    pr = average_precision_score(y_true, probs)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    # 💰 TRADE-OFF DE NEGÓCIO
    custo_fp = 100
    custo_fn = 500

    custo_total = (fp * custo_fp) + (fn * custo_fn)

    return acc, f1, roc, pr, custo_total, fp, fn

# ================================
# EXECUÇÃO + MLFLOW
# ================================
with mlflow.start_run(run_name="MLP_PyTorch"):

    model = MLP(X_train.shape[1]).to(DEVICE)

    model = train_model(model)

    acc, f1, roc, pr, custo, fp, fn = evaluate_model(model)

    # ========================
    # LOGGING
    # ========================
    mlflow.log_param("model", "MLP")
    mlflow.log_param("hidden_layers", [64, 32])
    mlflow.log_param("activation", "ReLU")
    mlflow.log_param("optimizer", "Adam")
    mlflow.log_param("learning_rate", 0.001)

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc)
    mlflow.log_metric("pr_auc", pr)

    mlflow.log_metric("business_cost", custo)
    mlflow.log_metric("false_positive", fp)
    mlflow.log_metric("false_negative", fn)

    mlflow.pytorch.log_model(model, "mlp_model")

    # ========================
    # OUTPUT
    # ========================
    print("\n===== RESULTADOS MLP =====")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1-score: {f1:.4f}")
    print(f"ROC-AUC: {roc:.4f}")
    print(f"PR-AUC: {pr:.4f}")
    print(f"Custo Total: {custo}")
    print(f"FP: {fp} | FN: {fn}")

2026/04/06 19:28:26 INFO mlflow.tracking.fluent: Experiment with name 'churn_project' does not exist. Creating a new experiment.
2026/04/06 19:28:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/06 19:28:27 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.



===== RESULTADOS MLP =====
Accuracy: 0.7885
F1-score: 0.5694
ROC-AUC: 0.8324
PR-AUC: 0.6083
Custo Total: 100600
FP: 121 | FN: 177


In [ ]:
# ================================
# IMPORTS
# ================================
import os

import mlflow
import mlflow.pytorch
import pandas as pd
import torch
import torch.nn as nn
from mlflow.tracking import MlflowClient
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ================================
# CONFIG MLFLOW GLOBAL
# ================================
BASE_DIR = os.path.abspath("..")

mlflow.set_tracking_uri(f"sqlite:///{BASE_DIR}/mlflow.db")
mlflow.set_registry_uri(f"sqlite:///{BASE_DIR}/mlflow.db")

mlflow.set_experiment("churn_project")

client = MlflowClient()

# ================================
# CONFIG TORCH
# ================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

# ================================
# DADOS
# ================================
dados = pd.read_csv(f"{BASE_DIR}/data/processed/Customer-Churn-processed.csv")

X = dados.drop('Churn', axis=1).values
y = dados['Churn'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# tensors
X_train = torch.tensor(X_train, dtype=torch.float32).to(DEVICE)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1).to(DEVICE)

X_test = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1,1).to(DEVICE)

# ================================
# MODELO
# ================================
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# ================================
# TREINO
# ================================
def train_model(model, epochs=100, patience=10):

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    best_loss = np.inf
    patience_counter = 0

    for epoch in range(epochs):

        model.train()
        optimizer.zero_grad()

        outputs = model(X_train)
        loss = criterion(outputs, y_train)

        loss.backward()
        optimizer.step()

        if loss.item() < best_loss:
            best_loss = loss.item()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping na época {epoch}")
            break

    return model

# ================================
# MÉTRICAS
# ================================
def evaluate_model(model):

    model.eval()
    with torch.no_grad():
        probs = model(X_test).cpu().numpy()

    y_pred = (probs > 0.5).astype(int)
    y_true = y_test.cpu().numpy()

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc = roc_auc_score(y_true, probs)
    pr = average_precision_score(y_true, probs)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    custo_fp = 100
    custo_fn = 500
    custo_total = (fp * custo_fp) + (fn * custo_fn)

    return acc, f1, roc, pr, custo_total

# ================================
# EXECUÇÃO + REGISTRY
# ================================
with mlflow.start_run(run_name="MLP_with_registry") as run:

    model = MLP(X_train.shape[1]).to(DEVICE)
    model = train_model(model)

    acc, f1, roc, pr, custo = evaluate_model(model)

    # ========================
    # LOGGING
    # ========================
    mlflow.log_param("model", "MLP")
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc)
    mlflow.log_metric("pr_auc", pr)
    mlflow.log_metric("business_cost", custo)

    # salva modelo
    mlflow.pytorch.log_model(model, "model")

    run_id = run.info.run_id
    model_uri = f"runs:/{run_id}/model"

    print("\nModelo logado em:", model_uri)

# ================================
# REGISTRAR NO MODEL REGISTRY
# ================================
model_name = "churn_mlp_model"

# cria registro (se não existir)
try:
    client.create_registered_model(model_name)
except Exception:
    print("Modelo já existe no registry")

# cria versão
model_version = client.create_model_version(
    name=model_name,
    source=model_uri,
    run_id=run_id
)

print(f"\nModelo registrado: {model_name} v{model_version.version}")

# ================================
# PROMOVER PARA STAGING
# ================================
client.transition_model_version_stage(
    name=model_name,
    version=model_version.version,
    stage="Staging"
)

print("Modelo promovido para STAGING")

# ================================
# (OPCIONAL) PROMOVER PARA PRODUÇÃO
# ================================
# client.transition_model_version_stage(
#     name=model_name,
#     version=model_version.version,
#     stage="Production"
# )

2026/04/06 19:42:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/06 19:42:48 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.



Modelo logado em: runs:/318e91684cda4542abcf3c51c8e3fc99/model

Modelo registrado: churn_mlp_model v1
Modelo promovido para STAGING


/tmp/ipykernel_3252628/1333726356.py:197: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(
